In [1]:
import numpy as np
from tensorflow import keras
import tensorflow as tf

tf.keras.backend.clear_session()

NUM_FEATURES = 21*8
NUM_CLASSES = 1001 #1000 + тишина

inputs = keras.layers.Input(shape=(None, NUM_FEATURES), ragged=True)
x = keras.layers.GRU(256, return_sequences=True, dropout=0.35, recurrent_dropout=0.2)(inputs)
x = keras.layers.GRU(128, return_sequences=False, dropout=0.35, recurrent_dropout=0.2)(x)  # Changed to False
x = keras.layers.Dense(64, activation='relu')(x)
x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = keras.Model(inputs=inputs, outputs=outputs)
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

/Users/iaroslav/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.13.0


In [2]:
import mediapipe as mp
import cv2
import numpy as np
import time

class GestureRecognizer:
    def __init__(self, model=model):
        self.hands = mp.solutions.hands.Hands()
        self.model = model
        self.landmarks_buffer = []
        self.prev_landmarks = None
        self.last_prediction_time = 0

    def extract_landmarks(self, frame):
        results = self.hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        
        if not results.multi_hand_landmarks:
            self.prev_landmarks = None
            return [0.0] * 168, frame
            
        hand_landmarks = results.multi_hand_landmarks[0]
        current_coords = []
        
        for landmark in hand_landmarks.landmark:
            current_coords.extend([landmark.x, landmark.y, landmark.z, landmark.visibility])
        
        if self.prev_landmarks is None:
            self.prev_landmarks = current_coords
            return current_coords + [0.0] * 84, frame
            
        changes = [curr - prev for curr, prev in zip(current_coords, self.prev_landmarks)]
        self.prev_landmarks = current_coords
        return current_coords + changes, frame

    def predict_online(self, frames_for_prediction=10, prediction_interval=0.1):
        cap = cv2.VideoCapture(0)
        
        while True:
            success, frame = cap.read()
            if not success:
                break
            
            current_time = time.time()
            landmarks, _ = self.extract_landmarks(frame)
            self.landmarks_buffer.append(landmarks)
            
            if (current_time - self.last_prediction_time >= prediction_interval and 
                len(self.landmarks_buffer) >= frames_for_prediction):
                
                sequence = np.array(self.landmarks_buffer[-frames_for_prediction:])
                prediction = self.model.predict(sequence.reshape(1, frames_for_prediction, -1), verbose=0)
                
                if prediction is not None:
                    predicted_class = np.argmax(prediction[0])
                    confidence = np.max(prediction[0])
                    print(f"Prediction: Class {predicted_class}, Confidence: {confidence:.2f}")
                
                self.last_prediction_time = current_time
                self.landmarks_buffer = self.landmarks_buffer[5:]
            
            cv2.imshow('Gesture Recognition', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        
        cap.release()
        cv2.destroyAllWindows()

In [4]:
import matplotlib.pyplot as plt


def plot_cm_with_matplotlib(y_true, y_pred, class_names):
    cm = plt.confusion_matrix(y_true, y_pred)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    ax.figure.colorbar(im, ax=ax)
    
    # Подписи осей
    ax.set(xticks=np.arange(cm.shape[1]),
           yticks=np.arange(cm.shape[0]),
           xticklabels=class_names,
           yticklabels=class_names,
           title='Confusion Matrix',
           ylabel='True Label',
           xlabel='Predicted Label')
    
    # Поворот подписей
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    
    # Добавление чисел в ячейки
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], 'd'),
                   ha="center", va="center",
                   color="white" if cm[i, j] > thresh else "black")
    
    plt.tight_layout()
    plt.show()
    
    return cm

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
# 
# data = np.load('landmarks.npy', allow_pickle=True).item()
# landmarks = data['landmarks']
# target = data['targets']
# video_train, video_test, target_train, target_test = train_test_split(
#     landmarks, 
#     target, 
#     test_size=0.15, #по 3 видео для проверки на каждый класс
#     random_state=42,
#     stratify=target
# )

# encoder = LabelEncoder()
# target_train_encoded = encoder.fit_transform(target_train)
# target_test_encoded = encoder.transform(target_test)
# sample_xtrai, sample_xtes, sample_ytrai, sample_ytes = [video_train[:10], 
#                                                             video_test[:10], 
#                                                             target_train_encoded[:10], 
#                                                             target_test_encoded[:10]]
# #

# def make_ragged(data):
#     try:

#         ragged_data = tf.ragged.constant(data)
#         print(f"Successfully created ragged tensor with shape: {ragged_data.shape}")
#         return ragged_data
#     except ValueError as e:
#         print(f"Conversion error: {e}")
#         print("Falling back to padding...")
#         from tensorflow.keras.preprocessing.sequence import pad_sequences
#         return pad_sequences(data, dtype='float32', padding='post')

# def target_for_ragged(data):
#     return tf.constant(data, dtype = tf.int32)

# sample_xtrain = make_ragged(sample_xtrai)
# sample_xtest = make_ragged(sample_xtes)
# sample_ytrain = target_for_ragged(sample_ytrai)
# sample_ytest = target_for_ragged(sample_ytes)

# video_train_ragged = make_ragged(video_train)
# video_test_ragged = make_ragged(video_test)

In [6]:
!pip install ijson
!pip install tqdm

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [9]:
import pandas as pd
import numpy as np
import ijson  # для потоковой обработки больших JSON
from tqdm import tqdm

path_annotations = 'data/annotations.csv'
path_landmarks = 'data/slovo_mediapipe.json'

print("1. Загрузка аннотаций...")
targets = pd.read_csv(path_annotations, sep='\t')
print("✅ Аннотации загружены")

print(f"Колонки: {targets.columns.tolist()}")
print(f"Размер: {targets.shape}")

# Создаем mapping из ID в текст
id_mapping = dict(zip(targets['attachment_id'].astype(str), targets['text']))
print(f"Создано {len(id_mapping)} соответствий ID->текст")

print("2. Потоковая загрузка landmarks...")

def process_large_json_streaming_corrected(file_path, id_mapping, max_sessions=None):
    """Обработка большого JSON файла потоково"""
    data = []  # ← Теперь список, а не словарь!
    processed_count = 0
    skipped_count = 0
    
    with open(file_path, 'r', encoding='utf-8') as f:
        sessions = ijson.kvitems(f, '')
        
        for session_id, frames in tqdm(sessions, desc="Обработка сессий"):
            if session_id in id_mapping:
                text = id_mapping[session_id]
                # Сохраняем как кортеж (text, frames)
                data.append((text, frames))
                processed_count += 1
            else:
                skipped_count += 1
            
            if max_sessions and processed_count >= max_sessions:
                break
    
    print(f"✅ Найдено соответствий: {processed_count}")
    print(f"❌ Пропущено (нет в аннотациях): {skipped_count}")
    return data, processed_count

# Перезагружаем данные правильно
data, matched_count = process_large_json_streaming_corrected(
    path_landmarks, 
    id_mapping
)

1. Загрузка аннотаций...
✅ Аннотации загружены
Колонки: ['attachment_id', 'text', 'user_id', 'height', 'width', 'length', 'train', 'begin', 'end']
Размер: (20400, 9)
Создано 20400 соответствий ID->текст
2. Потоковая загрузка landmarks...


Обработка сессий: 20000it [24:49, 13.42it/s] 


✅ Найдено соответствий: 20000
❌ Пропущено (нет в аннотациях): 0


In [11]:
def prepare_hand_data_corrected(data):
    y = []
    X = []
    
    print("Анализ данных...")
    frame_lengths = []
    
    for word, frames in data:  # Теперь data - список кортежей
        frame_lengths.append(len(frames))
        
        video = []
        for frame_data in frames:
            frame = []
            for hand_landmarks in frame_data.values():
                for point in hand_landmarks:
                    frame.extend([
                        float(point['x']),
                        float(point['y']),
                        float(point['z'])
                    ])
            video.append(frame)
        
        y.append(word)
        X.append(video)
    
    print(f"Статистика по кадрам:")
    print(f"  Всего видео: {len(X)}")
    print(f"  Мин. кадров: {min(frame_lengths)}")
    print(f"  Макс. кадров: {max(frame_lengths)}")
    print(f"  Сред. кадров: {np.mean(frame_lengths):.1f}")
    
    return np.array(X, dtype='object'), y  # Используем object из-за разной длины

X, y = prepare_hand_data_corrected(data)

Анализ данных...
Статистика по кадрам:
  Всего видео: 20000
  Мин. кадров: 0
  Макс. кадров: 236
  Сред. кадров: 46.8


In [13]:
np.save('new_handmarks.npy', X, allow_pickle=True)
np.save('new_targets.npy', y, allow_pickle=True)

In [4]:
X = np.load('new_handmarks.npy', allow_pickle=True)
y = np.load('new_targets.npy', allow_pickle=True)

In [9]:
from keras.preprocessing.sequence import pad_sequences

# 1. Сначала убедимся, что все кадры имеют одинаковую длину
print("Проверка структуры данных...")
for i, video in enumerate(X):
    frame_lengths = [len(frame) for frame in video]
    if len(set(frame_lengths)) > 1:
        print(f"Видео {i}: кадры разной длины - {frame_lengths}")
        break

# 2. Найдем максимальную длину кадра
max_frame_len = max(len(frame) for video in X for frame in video)
print(f"Максимальная длина кадра: {max_frame_len}")

# 3. Дополним каждый кадр до одинаковой длины
X_padded_frames = []
for video in X:
    padded_video = []
    for frame in video:
        # Дополняем каждый кадр нулями до max_frame_len
        if len(frame) < max_frame_len:
            padded_frame = frame + [0.0] * (max_frame_len - len(frame))
        else:
            padded_frame = frame[:max_frame_len]
        padded_video.append(padded_frame)
    X_padded_frames.append(padded_video)

# 4. Теперь применяем pad_sequences к видео
X_padded = pad_sequences(X_padded_frames, maxlen=max_frame_len, dtype='float32', 
                        padding='post', truncating='post')
X = np.array(X_padded, dtype=np.float32)
print(f"Форма после padding: {X.shape}")  # (видео, кадры, признаки)

# 5. Разделяем данные
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.15, random_state=42, stratify=y
)

# 6. Кодируем метки
encoder = LabelEncoder()
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)

print("Готово! Данные подготовлены.")

Проверка структуры данных...
Видео 2: кадры разной длины - [126, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63, 63]
Максимальная длина кадра: 189
Форма после padding: (20000, 189, 189)
Готово! Данные подготовлены.


In [28]:
import tensorflow as tf
from tensorflow import keras

# Полная очистка
tf.keras.backend.clear_session()
tf.compat.v1.reset_default_graph()

# СУПЕР простая модель для теста
simple_model = keras.Sequential([
    keras.layers.Input(shape=X_train.shape[1:]),  # автоматически подстроимся под форму
    keras.layers.Flatten(),  # если данные 2D/3D
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(NUM_CLASSES, activation='softmax')
])

simple_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("=== МОДЕЛЬ ===")
simple_model.summary()

=== МОДЕЛЬ ===
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 35721)             0         
                                                                 
 dense (Dense)               (None, 64)                2286208   
                                                                 
 dense_1 (Dense)             (None, 1001)              65065     
                                                                 
Total params: 2351273 (8.97 MB)
Trainable params: 2351273 (8.97 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
# Самый простой вариант - запустить на 2-3 эпохах
try:
    history = model_new.fit(X_train, 
                        y_train_encoded, 
                        validation_data=(X_test, y_test_encoded),
                        epochs=3,  # всего 3 эпохи для теста
                        batch_size=32,
                        verbose=1)
    print("Обучение завершено успешно!")
except Exception as e:
    print(f"Ошибка: {e}")

Epoch 1/3
94/94 [==============================] - ETA: 0s - loss: 6.9471 - accuracy: 6.6667e-04